# 01. Exploration of Raw Data.

I've retreived the blocks and transactions using Google Big Query to used for training data to build our Unsupervised cluster in order to inspect what type of patterns arise, and get some information about the data. 

* 1 year worth of blocks (and its transactions), and transfer event logs were pulled from BigQuery's public dataset then fetched into local storage in paraquet format for downstream features.


* First with authenticate with `gcloud auth application-default login`


## Raw Block entries (Big Query vs. Alchemy via RPC)

Requires authenticating using `gcloud auth application-default login` first. I am generating two dataframes to demonstrate side by side.

In [108]:
# import relevant libraries and set up BigQuery client and RPC function
from google.cloud import bigquery
client = bigquery.Client(project="chainsense-497110")

import requests, os, dotenv, pandas as pd
dotenv.load_dotenv()
def rpc(method, params):
    url = f"https://eth-mainnet.g.alchemy.com/v2/{os.getenv("ALCHEMY_KEY")}"
    r = requests.post(url, json={"jsonrpc": "2.0", "id":1, "method":method, "params":params})
    return r.json()["result"]

In [113]:
sample = client.query("""
    SELECT * FROM `bigquery-public-data.crypto_ethereum.blocks`
    ORDER BY number DESC
    LIMIT 1
""").to_dataframe()
bq_df = (
    sample.T # Take 1 row sample (column major) and convert into row major
    .reset_index() # after transpose, the original column names are now the index of the dataframe
    .rename(columns={"index": "field", 0: "value"}) # 0 because after transpose the single row had label 0
    .assign(type=lambda df: df["value"].apply(lambda v: type(v).__name__))
    [["field", "type", "value"]] # reorder columns to be more intuitive
)
block = rpc("eth_getBlockByNumber", ["latest", True])
rpc_df = pd.DataFrame([(k,type(v).__name__,v) for k,v in block.items()], columns=["field", "type", "value"])

/Users/mr.po/Documents/GitHub/ucsd-ChainSense/.venv/lib/python3.13/site-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


In [ ]:
# Big Query Blocks vs RPC (eth_getBlockByNumber) Blocks
pd.concat([bq_df, rpc_df], axis=1)

,field,type,value,field,type,value
0,timestamp,Timestamp,2026-05-22 23:59:59+00:00,hash,str,0x09024c13961b37ce98856b716cd7e309ef26fe637025...
1,number,int,25154243,parentHash,str,0x42984a35000eaaab94a63c31cab448ec0a3383275169...
2,hash,str,0x26016d681996792f2d2d39863b7f100b1b937b4c5e37...,sha3Uncles,str,0x1dcc4de8dec75d7aab85b567b6ccd41ad312451b948a...
3,parent_hash,str,0x901f793f099117eb211cf5977a1af393dd4e13469191...,miner,str,0xdadb0d80178819f2319190d340ce9a924f783711
4,nonce,str,0x0000000000000000,stateRoot,str,0xbbab1571b37f18a95d86107fd601683bd8207635f9ab...
5,sha3_uncles,str,0x1dcc4de8dec75d7aab85b567b6ccd41ad312451b948a...,transactionsRoot,str,0xf3fc685c16050f817d35b460972d16d66a8b0f93bcfa...
6,logs_bloom,str,0xbdab57c5f7d06dd87ffd7c3ee2eb96dfdfd5fc4eb677...,receiptsRoot,str,0x75740c9c2b6b7eab2f0e66cf2895e55224cc9fe23112...
7,transactions_root,str,0x01f1ade969acfebb8e0d94b88e6618e2fc7fcce3cb91...,logsBloom,str,0x87f173cb67fbfffd71d77fffbccff45b732da8fb048d...
8,state_root,str,0x3c2e0634907eb91659d9d70c11d38ac6a3ad5f0d06af...,difficulty,str,0x0
9,receipts_root,str,0xe9ff53d1e2f06495acf0bdf93b218dac7a5e1877bfbf...,number,str,0x17fde66


## Raw Tx entries (Big Query vs. Alchemy via RPC)

In [ ]:
bq_tx = client.query("""
    SELECT * FROM `bigquery-public-data.crypto_ethereum.transactions`
    WHERE DATE(block_timestamp) = CURRENT_DATE() - 1
    LIMIT 1
""").to_dataframe()
bq_tx = bq_tx.T.reset_index().rename(columns={"index": "field", 0: "value"}).assign(type=lambda df: df["value"].apply(lambda v: type(v).__name__))[["field", "type", "value"]]
rpc_tx = block["transactions"][0]
rpc_tx = pd.DataFrame([(k,type(v).__name__,v) for k,v in rpc_tx.items()], columns=["field", "type", "value"])

/Users/mr.po/Documents/GitHub/ucsd-ChainSense/.venv/lib/python3.13/site-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


In [121]:
pd.concat([bq_tx, rpc_tx], axis=1)

,field,type,value,field,type,value
0,hash,str,0x0d259a89d7d22858d988a1ccb65f73a55ea31882fdac...,type,str,0x2
1,nonce,int,50073,chainId,str,0x1
2,transaction_index,int,102,nonce,str,0x8c27
3,from_address,str,0x6402076597364f810bb553b85d67d597df92d823,gas,str,0x88b8
4,to_address,str,0x45118b56d5066a6352d78d15f795cfab33c4def3,maxFeePerGas,str,0x27f9b8f4
5,value,Decimal,0E-9,maxPriorityFeePerGas,str,0x1cfadc84
6,gas,int,36547,to,str,0x681e908b8ab57c49c74d770f369754ccc3e1ae09
7,gas_price,int,195370959,value,str,0x0
8,input,str,0xe4715bb5000000000000000000000000000000000000...,accessList,list,[]
9,receipt_cumulative_gas_used,int,5775443,input,str,0x6a117b3186c930b461005de552725c0005f490e30700...


## Raw Log entries (Big Query vs. Alchemy via RPC)

In [2]:
import pandas as pd

def inspect(path):
    df = pd.read_parquet(path)
    row = df.iloc[0]
    return pd.DataFrame({
        "field": row.index,
        "type": [type(v).__name__ for v in row.values],
        "value": row.values,
    })

inspect("../data/bronze/token_transfers/transfers-000000000000.parquet")

,field,type,value
0,token_address,str,0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2
1,from_address,str,0x7a250d5630b4cf539739df2c5dacb4c659f2488d
2,to_address,str,0x6b4207a321319d5740d2375471953ac95d803598
3,value,str,10072850674802443
4,transaction_hash,str,0x991fa1b617f4b525881aa6c91b61b565ccddd35168ee...
5,block_number,int64,23386241
6,block_timestamp,Timestamp,2025-09-18 00:00:11


# Contracts

BQ supports 

In [5]:
inspect("../data/bronze/contracts/contracts-000000000000.parquet")

,field,type,value
0,address,str,0xdc3c18e1a832e8fed1039e4a8e6100e24579105d


In [6]:
inspect("../live/bronze/blocks/date=2026-05-24/25163189.parquet")

,field,type,value
0,block_number,int64,25163189
1,block_timestamp,Timestamp,2026-05-24 05:55:11+00:00
2,block_hash,str,de650d36d20b4d948f6af7d42a33ab5d1d0fdc52b01b40...
3,base_fee_per_gas,int64,54129178


In [8]:
%pwd

'/Users/mr.po/Documents/GitHub/ucsd-ChainSense/notebooks'

In [10]:
import duckdb

duckdb.sql("""
    COPY (SELECT * FROM '../data/bronze/contracts/contracts-*.parquet')
    TO '../data/bronze/contracts.parquet'
    (FORMAT PARQUET, COMPRESSION ZSTD)
""")

In [17]:
import duckdb
con = duckdb.connect()
print(con.execute("DESCRIBE SELECT * FROM read_parquet('../data/bronze/transactions/transactions-000000000000.parquet')").df())

                column_name column_type null   key default extra
0                   tx_hash     VARCHAR  YES  None    None  None
1              block_number      BIGINT  YES  None    None  None
2           block_timestamp   TIMESTAMP  YES  None    None  None
3              from_address     VARCHAR  YES  None    None  None
4                to_address     VARCHAR  YES  None    None  None
5                 value_wei     VARCHAR  YES  None    None  None
6                 gas_limit      BIGINT  YES  None    None  None
7             gas_price_wei     VARCHAR  YES  None    None  None
8                  gas_used      BIGINT  YES  None    None  None
9                    status      BIGINT  YES  None    None  None
10  effective_gas_price_wei     VARCHAR  YES  None    None  None
11          method_selector     VARCHAR  YES  None    None  None


In [23]:
inspect("../data/bronze/transactions/transactions-000000000000.parquet")

,field,type,value
0,tx_hash,str,0x6c750dbf01abe4b8193cf3e7951e10010de1814f5ea0...
1,block_number,int64,24984754
2,block_timestamp,Timestamp,2026-04-29 09:09:47
3,from_address,str,0x974caa59e49682cda0ad2bbe82983419a2ecc400
4,to_address,str,0xe202955eaa74b79a0148d01c108cb1f2ebb29b24
5,value_wei,str,2485224640500000
6,gas_limit,int64,21000
7,gas_price_wei,str,1183440305
8,gas_used,int64,21000
9,status,int64,1


## Partitioning Txs

In [24]:
# partition_transactions.py
import duckdb
from pathlib import Path

SRC = "../data/bronze/transactions/transactions-*.parquet"
DST = "../data/silver/transactions"

Path(DST).mkdir(parents=True, exist_ok=True)

con = duckdb.connect()

# Tune for M4 Mini. Adjust threads/memory to your machine.
con.execute("PRAGMA threads=8;")
con.execute("PRAGMA memory_limit='10GB';")
con.execute("PRAGMA temp_directory='../data/duckdb_tmp';")
con.execute("PRAGMA preserve_insertion_order=false;")  # faster, order doesn't matter here

con.execute(f"""
    COPY (
        SELECT
            tx_hash,
            block_number,
            block_timestamp,
            from_address,
            to_address,
            value_wei,
            gas_limit,
            gas_used,
            gas_price_wei,
            effective_gas_price_wei,
            status,
            method_selector,
            CAST(strftime(block_timestamp, '%Y') AS INTEGER) AS year,
            CAST(strftime(block_timestamp, '%m') AS INTEGER) AS month
        FROM read_parquet('{SRC}')
    )
    TO '{DST}'
    (FORMAT PARQUET,
     PARTITION_BY (year, month),
     COMPRESSION ZSTD,
     OVERWRITE_OR_IGNORE);
""")

print("Done. Partitions:")
for p in sorted(Path(DST).glob("year=*/month=*")):
    size_gb = sum(f.stat().st_size for f in p.rglob("*.parquet")) / 1e9
    print(f"  {p.name}: {size_gb:.2f} GB")

print("\nRow counts:")
result = con.execute(f"""
    SELECT year, month, COUNT(*) AS tx_count
    FROM read_parquet('{DST}/**/*.parquet', hive_partitioning=true)
    GROUP BY year, month
    ORDER BY year, month
""").fetchall()

for year, month, count in result:
    print(f"  {year}-{month:02d}: {count:,} txs")

Done. Partitions:
  month=10: 4.08 GB
  month=11: 3.84 GB
  month=12: 4.29 GB
  month=5: 1.20 GB
  month=6: 3.65 GB
  month=7: 4.04 GB
  month=8: 4.48 GB
  month=9: 4.18 GB
  month=1: 5.98 GB
  month=2: 5.34 GB
  month=3: 5.91 GB
  month=4: 6.32 GB
  month=5: 3.70 GB

Row counts:
  2025-05: 13,855,812 txs
  2025-06: 41,950,663 txs
  2025-07: 46,669,810 txs
  2025-08: 51,770,534 txs
  2025-09: 48,896,815 txs
  2025-10: 48,010,794 txs
  2025-11: 45,672,422 txs
  2025-12: 51,471,180 txs
  2026-01: 70,009,743 txs
  2026-02: 61,889,998 txs
  2026-03: 68,542,985 txs
  2026-04: 72,827,602 txs
  2026-05: 44,194,750 txs


Partiotioning blocks

In [27]:
inspect("../data/bronze/blocks/blocks.parquet")

,field,type,value
0,block_number,int64,22534698
1,block_timestamp,Timestamp,2025-05-22 00:00:11
2,base_fee_per_gas_wei,str,1263015985
3,miner,str,0x396343362be2a4da1ce0c1c210945346fb82aa49
4,block_gas_used,int64,18392433
5,block_gas_limit,int64,36000000
6,transaction_count,int64,253


In [26]:
"""
Partition bronze blocks parquet into silver, matching the transactions partition scheme
(year=YYYY/month=MM), keeping only block_number and base_fee_per_gas_wei.

Input:  ../data/bronze/blocks/blocks.parquet  (or blocks-*.parquet if sharded)
Output: ../data/silver/blocks/year=YYYY/month=MM/*.parquet
"""

import duckdb
from pathlib import Path

BRONZE_GLOB = "../data/bronze/blocks/blocks*.parquet"  # matches single file or shards
SILVER_DIR  = "../data/silver/blocks"

Path(SILVER_DIR).mkdir(parents=True, exist_ok=True)

con = duckdb.connect()
con.execute("PRAGMA threads=8")
con.execute("PRAGMA memory_limit='10GB'")
con.execute("PRAGMA temp_directory='../data/duckdb_tmp'")

query = f"""
COPY (
    SELECT
        block_number,
        base_fee_per_gas_wei,
        -- block_timestamp,  -- uncomment if you want to keep it in the silver file
        CAST(strftime(block_timestamp, '%Y') AS INTEGER) AS year,
        CAST(strftime(block_timestamp, '%m') AS INTEGER) AS month
    FROM read_parquet('{BRONZE_GLOB}')
)
TO '{SILVER_DIR}'
(
    FORMAT PARQUET,
    PARTITION_BY (year, month),
    OVERWRITE_OR_IGNORE,
    COMPRESSION ZSTD,
    ROW_GROUP_SIZE 100000
);
"""

con.execute(query)
print("Done. Output written to", SILVER_DIR)

# Sanity check
result = con.execute(f"""
    SELECT year, month, COUNT(*) AS block_count, MIN(block_number) AS min_blk, MAX(block_number) AS max_blk
    FROM read_parquet('{SILVER_DIR}/**/*.parquet', hive_partitioning=true)
    GROUP BY year, month
    ORDER BY year, month
""").fetchall()

for row in result:
    print(f"  {row[0]}-{row[1]:02d}: {row[2]:,} blocks  (block {row[3]:,}{row[4]:,})")

Done. Output written to ../data/silver/blocks
  2025-05: 71,445 blocks  (block 22,534,69822,606,142)
  2025-06: 214,531 blocks  (block 22,606,14322,820,673)
  2025-07: 221,840 blocks  (block 22,820,67423,042,513)
  2025-08: 222,052 blocks  (block 23,042,51423,264,565)
  2025-09: 214,678 blocks  (block 23,264,56623,479,243)
  2025-10: 221,523 blocks  (block 23,479,24423,700,766)
  2025-11: 214,154 blocks  (block 23,700,76723,914,920)
  2025-12: 221,132 blocks  (block 23,914,92124,136,052)
  2026-01: 222,240 blocks  (block 24,136,05324,358,292)
  2026-02: 200,575 blocks  (block 24,358,29324,558,867)
  2026-03: 222,159 blocks  (block 24,558,86824,781,026)
  2026-04: 215,341 blocks  (block 24,781,02724,996,367)
  2026-05: 150,704 blocks  (block 24,996,36825,147,071)


# Token Transfers

In [ ]:
inspect("../data/bronze/token_transfers/transfers-000000000000.parquet")


,field,type,value
0,token_address,str,0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2
1,from_address,str,0x7a250d5630b4cf539739df2c5dacb4c659f2488d
2,to_address,str,0x6b4207a321319d5740d2375471953ac95d803598
3,value,str,10072850674802443
4,transaction_hash,str,0x991fa1b617f4b525881aa6c91b61b565ccddd35168ee...
5,block_number,int64,23386241
6,block_timestamp,Timestamp,2025-09-18 00:00:11


In [31]:
"""
Partition bronze token_transfers parquet files into silver, matching the transactions
partition scheme (year=YYYY/month=MM).

Input:  ../data/bronze/token_transfers/transfers-*.parquet
Output: ../data/silver/token_transfers/year=YYYY/month=MM/*.parquet
"""

import duckdb
from pathlib import Path

BRONZE_GLOB = "../data/bronze/token_transfers/transfers-*.parquet"
SILVER_DIR  = "../data/silver/token_transfers"

Path(SILVER_DIR).mkdir(parents=True, exist_ok=True)

con = duckdb.connect()
con.execute("PRAGMA threads=8")
con.execute("PRAGMA memory_limit='12GB'")
con.execute("PRAGMA temp_directory='../data/duckdb_tmp'")

query = f"""
COPY (
    SELECT
        token_address,
        from_address,
        to_address,
        value,                   -- uint256, kept as string for precision
        transaction_hash,
        block_number,
        block_timestamp,
        CAST(strftime(block_timestamp, '%Y') AS INTEGER) AS year,
        CAST(strftime(block_timestamp, '%m') AS INTEGER) AS month
    FROM read_parquet('{BRONZE_GLOB}')
)
TO '{SILVER_DIR}'
(
    FORMAT PARQUET,
    PARTITION_BY (year, month),
    OVERWRITE_OR_IGNORE,
    COMPRESSION ZSTD,
    ROW_GROUP_SIZE 100000
);
"""

con.execute(query)
print("Done. Output written to", SILVER_DIR)

# Sanity check
result = con.execute(f"""
    SELECT year, month,
           COUNT(*) AS transfer_count,
           COUNT(DISTINCT token_address) AS unique_tokens
    FROM read_parquet('{SILVER_DIR}/**/*.parquet', hive_partitioning=true)
    GROUP BY year, month
    ORDER BY year, month
""").fetchall()

for row in result:
    print(f"  {row[0]}-{row[1]:02d}: {row[2]:>12,} transfers, {row[3]:>7,} unique tokens")

Done. Output written to ../data/silver/token_transfers
  2025-05:   17,272,110 transfers,  40,235 unique tokens
  2025-06:   51,381,284 transfers,  80,478 unique tokens
  2025-07:   59,967,299 transfers,  86,043 unique tokens
  2025-08:   83,150,340 transfers,  91,307 unique tokens
  2025-09:   92,704,686 transfers,  86,827 unique tokens
  2025-10:   71,917,101 transfers,  81,971 unique tokens
  2025-11:   75,307,061 transfers,  83,054 unique tokens
  2025-12:  144,657,353 transfers,  86,018 unique tokens
  2026-01:   99,879,448 transfers,  82,394 unique tokens
  2026-02:   98,880,044 transfers,  76,713 unique tokens
  2026-03:  113,039,329 transfers,  99,383 unique tokens
  2026-04:  110,279,743 transfers, 199,241 unique tokens
  2026-05:   70,238,323 transfers, 124,821 unique tokens


# Contracts
merge into single file, dedupe addresses

In [ ]:
inspect("/Users/mr.po/Documents/GitHub/ucsd-ChainSense/data/silver/contracts/contracts.parquet")

In [32]:
"""
Prepare contracts lookup table.
Input:  ../data/bronze/contracts/contracts-*.parquet
Output: ../data/silver/contracts/contracts.parquet  (single file, sorted, deduped)
"""

import duckdb
from pathlib import Path

BRONZE_GLOB = "../data/bronze/contracts/contracts*.parquet"
SILVER_DIR  = "../data/silver/contracts"
SILVER_FILE = f"{SILVER_DIR}/contracts.parquet"

Path(SILVER_DIR).mkdir(parents=True, exist_ok=True)

con = duckdb.connect()
con.execute("PRAGMA threads=8")
con.execute("PRAGMA memory_limit='12GB'")

query = f"""
COPY (
    SELECT DISTINCT
        lower(address) AS address   -- normalize case (defensive; BQ exports are lowercase)
    FROM read_parquet('{BRONZE_GLOB}')
    WHERE address IS NOT NULL
    ORDER BY address                -- sorted for better compression + faster joins
)
TO '{SILVER_FILE}'
(
    FORMAT PARQUET,
    COMPRESSION ZSTD,
    ROW_GROUP_SIZE 100000
);
"""

con.execute(query)

# Sanity check
n_contracts = con.execute(f"SELECT COUNT(*) FROM '{SILVER_FILE}'").fetchone()[0]
file_size_mb = Path(SILVER_FILE).stat().st_size / 1024 / 1024
print(f"Wrote {n_contracts:,} unique contract addresses to {SILVER_FILE} ({file_size_mb:.1f} MB)")

Wrote 99,392,431 unique contract addresses to ../data/silver/contracts/contracts.parquet (1968.8 MB)
